# LatentMAS-interp — GPU Test Suite

Pulls the repo, installs deps, runs logic tests + end-to-end smoke on real GPU.

In [ ]:
# 1. Clone / update repo
import os
REPO = "https://github.com/spraldev/LatentMAS-interp"
REPO_DIR = "/workspace/LatentMAS-interp"

if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} pull
else:
    !git clone {REPO} {REPO_DIR}

os.chdir(REPO_DIR)
print("cwd:", os.getcwd())

In [ ]:
# 2. Install deps
!pip install -q torch transformers accelerate datasets tqdm numpy

In [ ]:
# 3. Confirm GPU is visible
import torch
assert torch.cuda.is_available(), "No GPU found — wrong runtime?"
print(f"GPU: {torch.cuda.get_device_name(0)}  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# 4. Logic unit tests (no model, pure math/IO — should be <30s)
!python test_pipeline.py -v

In [ ]:
# 5. End-to-end smoke on GPU — downloads Qwen3-0.6B once (~2 GB), runs 5 ex/task (~5 min)
# This is the real test: actual model forward pass, W_a, KV capture, bucket assignment
import os, tempfile
SMOKE_DIR = "/workspace/runs/smoke"
os.makedirs(SMOKE_DIR, exist_ok=True)

!python final_run.py \
    --output_dir {SMOKE_DIR} \
    --conditions latent_mas single_agent_latent_greedy \
    --tasks gsm8k \
    --smoke --test

In [ ]:
# 6. Check smoke outputs
import json
from pathlib import Path

smoke = Path(SMOKE_DIR)

# W_a sanity
import torch
wa = torch.load(smoke / "wa_matrix.pt", weights_only=False)["W_a"].float()
S = torch.linalg.svdvals(wa)
frob = (wa - torch.eye(wa.shape[0])).norm().item()
print(f"W_a shape: {tuple(wa.shape)}")
print(f"cond number: {S.max()/S.min():.4f}  (want >>1 on Qwen3-4B, =1 on 0.6B/1.7B)")
print(f"Frob dist from identity: {frob:.4f}  (want >>1 on Qwen3-4B)")
print()

# example outputs
for cond in ["latent_mas", "single_agent_latent_greedy"]:
    metas = list((smoke / cond / "gsm8k").glob("example_*/metadata.json"))
    correct = sum(json.loads(m.read_text()).get("correct", False) for m in metas)
    print(f"{cond}: {correct}/{len(metas)} correct")

# buckets
bucket_file = smoke / "buckets" / "gsm8k.json"
if bucket_file.exists():
    from collections import Counter
    rows = json.loads(bucket_file.read_text())
    c = Counter(r["bucket"] for r in rows)
    print(f"\nBuckets: B1={c[1]} B2={c[2]} B3={c[3]} B4={c[4]} (n={len(rows)})")
    print(f"Bucket-1 rate: {c[1]/len(rows)*100:.1f}%  (need >0 on real 4B run)")
else:
    print("No bucket file yet — run both conditions first")

In [ ]:
# 7. RISK 1: MBPP+ subprocess scoring
# exec() runs in a worker process — most likely full-run failure point.
# Smoke: 3 examples of mbppplus, check no subprocess crash.
MBPP_SMOKE_DIR = "/workspace/runs/smoke_mbpp"
os.makedirs(MBPP_SMOKE_DIR, exist_ok=True)

!python final_run.py \
    --output_dir {MBPP_SMOKE_DIR} \
    --conditions latent_mas \
    --tasks mbppplus \
    --smoke --test

# Check results
mbpp_metas = list(Path(MBPP_SMOKE_DIR).glob("latent_mas/mbppplus/example_*/metadata.json"))
print(f"MBPP+ examples completed: {len(mbpp_metas)}  (want >0, no crash = pass)")
for m in sorted(mbpp_metas):
    d = json.loads(m.read_text())
    print(f"  {m.parent.name}: correct={d.get('correct')}  prediction_len={len(str(d.get('prediction','')))}")

In [ ]:
# 8. RISK 2: OOM / resume
# Simulate interrupted run: run 3 examples, then re-run — must skip completed examples
# and not OOM on the second pass.
OOM_SMOKE_DIR = "/workspace/runs/smoke_resume"
os.makedirs(OOM_SMOKE_DIR, exist_ok=True)

# First pass: 3 examples
!python final_run.py \
    --output_dir {OOM_SMOKE_DIR} \
    --conditions latent_mas \
    --tasks gsm8k \
    --smoke --test

first_pass = list(Path(OOM_SMOKE_DIR).glob("latent_mas/gsm8k/example_*/metadata.json"))
print(f"First pass: {len(first_pass)} examples written")

# Second pass: same command — must reuse existing outputs
import time
t0 = time.time()
!python final_run.py \
    --output_dir {OOM_SMOKE_DIR} \
    --conditions latent_mas \
    --tasks gsm8k \
    --smoke --test
elapsed = time.time() - t0

second_pass = list(Path(OOM_SMOKE_DIR).glob("latent_mas/gsm8k/example_*/metadata.json"))
print(f"Second pass: {len(second_pass)} examples  elapsed={elapsed:.1f}s")
print(f"Resume working: {elapsed < 60}  (second pass should be <60s if skipping completed examples)")

In [ ]:
# 9. RISK 3: Storage budget
# Check disk usage of smoke runs and project to full-run size.
import shutil

def dir_size_gb(path):
    total = sum(f.stat().st_size for f in Path(path).rglob("*") if f.is_file())
    return total / 1e9

smoke_gb = dir_size_gb(SMOKE_DIR)
n_smoke_examples = 5  # --smoke runs 5 examples
n_full_examples = 200
n_conditions_smoke = 2
n_conditions_full = 17
n_tasks_full = 3

# Scale from smoke (5 ex, 2 cond, 1 task) to full (200 ex, 17 cond, 3 tasks)
scale = (n_full_examples / n_smoke_examples) * (n_conditions_full / n_conditions_smoke) * n_tasks_full
projected_gb = smoke_gb * scale

print(f"Smoke disk usage:    {smoke_gb:.2f} GB  ({n_smoke_examples} ex, {n_conditions_smoke} cond, 1 task)")
print(f"Projected full run:  {projected_gb:.1f} GB  ({n_full_examples} ex, {n_conditions_full} cond, {n_tasks_full} tasks)")
print()

total_disk_gb = shutil.disk_usage("/workspace").total / 1e9
free_disk_gb = shutil.disk_usage("/workspace").free / 1e9
print(f"Volume total: {total_disk_gb:.0f} GB   free: {free_disk_gb:.0f} GB")
if projected_gb > free_disk_gb * 0.8:
    print("WARNING: projected usage exceeds 80% of free space — consider --no_layer_hidden or --no_kv_latent")
else:
    print("Storage OK")

In [ ]:
# 10. RISK 4: Untested conditions — topk_gated, random_gated, exp_m_identity_wa
# These share the same core machinery but have never run in smoke.
# Run 3 examples each on gsm8k to confirm no crash.
COND_SMOKE_DIR = "/workspace/runs/smoke_conds"
os.makedirs(COND_SMOKE_DIR, exist_ok=True)

!python final_run.py \
    --output_dir {COND_SMOKE_DIR} \
    --conditions topk_gated random_gated exp_m_identity_wa \
    --tasks gsm8k \
    --smoke --test

print("\n--- Untested condition results ---")
for cond in ["topk_gated", "random_gated", "exp_m_identity_wa"]:
    metas = list(Path(COND_SMOKE_DIR).glob(f"{cond}/gsm8k/example_*/metadata.json"))
    if not metas:
        print(f"  {cond}: NO OUTPUT — crashed or condition name wrong")
        continue
    correct = sum(json.loads(m.read_text()).get("correct", False) for m in metas)
    print(f"  {cond}: {len(metas)} examples completed, {correct}/{len(metas)} correct")